# 10 — Final Smart Manufacturing AI Integration

## Purpose

This notebook demonstrates the **complete end-to-end AI pipeline** developed in this project.

It does not retrain the ML models or regenerate all historical LLM/RAG reports. Instead, it uses the completed project artifacts to demonstrate how one actionable runtime event can flow through:

```text
IoT / Runtime Event
        ↓
Smart Twin / Agent Decision
        ↓
Orchestrator
        ↓
Machine Memory
        ↓
RAG Knowledge Retrieval
        ↓
TCS MaaS LLM
        ↓
Final Maintenance Intelligence
```

### What this proves

The project is not just a collection of separate notebooks. The layers can be connected into a single decision-support workflow:

- **Smart Twin / ML** supplies predictive evidence.
- **Maintenance and Alert Agents** supply operational decisions.
- **Orchestrator** coordinates actionable runtime events.
- **Memory** supplies the machine's historical context.
- **RAG** supplies relevant maintenance/domain knowledge.
- **LLM** synthesizes the supplied information into a human-readable report.

### Important

The deterministic ML/agent/orchestrator decisions remain authoritative. The LLM is an explanation and intelligence layer; it must not invent or override those decisions.


In [1]:
# ============================================
# STEP 1: IMPORTS AND PROJECT PATHS
# ============================================

from pathlib import Path
import os
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Final integration environment ready.")
print("Working directory:", Path.cwd())


Final integration environment ready.
Working directory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/notebooks


In [2]:
# ============================================
# STEP 2: LOCATE COMPLETED PROJECT ARTIFACTS
# ============================================

possible_roots = [
    Path.cwd(),
    Path.cwd().parent
]

project_root = None

for root in possible_roots:
    if (
        (root / "data" / "processed").exists()
        and (root / "notebooks").exists()
    ):
        project_root = root
        break

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root."
    )

processed_dir = project_root / "data" / "processed"
knowledge_dir = project_root / "data" / "knowledge"

orchestrator_path = processed_dir / "orchestrated_runtime_events.csv"
memory_path = processed_dir / "machine_memory.csv"
memory_summary_path = processed_dir / "machine_memory_summary.csv"
llm_path = processed_dir / "llm_maintenance_reports.csv"
rag_path = processed_dir / "rag_maintenance_reports.csv"
knowledge_path = knowledge_dir / "maintenance_knowledge_base.json"

required_artifacts = {
    "orchestrator": orchestrator_path,
    "memory": memory_path,
    "memory_summary": memory_summary_path,
    "llm_reports": llm_path,
    "rag_reports": rag_path,
    "knowledge_base": knowledge_path,
}

missing_artifacts = [
    name
    for name, path in required_artifacts.items()
    if not path.exists()
]

if missing_artifacts:
    raise FileNotFoundError(
        "Missing completed artifacts: "
        + ", ".join(missing_artifacts)
    )

print("All completed project artifacts found.")
for name, path in required_artifacts.items():
    print(f"{name}: {path}")


All completed project artifacts found.
orchestrator: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/orchestrated_runtime_events.csv
memory: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory.csv
memory_summary: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/machine_memory_summary.csv
llm_reports: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/llm_maintenance_reports.csv
rag_reports: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/rag_maintenance_reports.csv
knowledge_base: /home/labuser/Desktop/Project/3203141-Capstone_Proj/data/knowledge/maintenance_knowledge_base.json


In [3]:
# ============================================
# STEP 3: LOAD COMPLETED ARTIFACTS
# ============================================

orchestrated_df = pd.read_csv(orchestrator_path)
memory_df = pd.read_csv(memory_path)
memory_summary_df = pd.read_csv(memory_summary_path)
llm_reports_df = pd.read_csv(llm_path)
rag_reports_df = pd.read_csv(rag_path)

print("Orchestrated events:", orchestrated_df.shape)
print("Machine memory:", memory_df.shape)
print("Machine summaries:", memory_summary_df.shape)
print("LLM reports:", llm_reports_df.shape)
print("RAG reports:", rag_reports_df.shape)


Orchestrated events: (1930, 15)
Machine memory: (55, 28)
Machine summaries: (7, 16)
LLM reports: (55, 12)
RAG reports: (55, 14)


In [4]:
# ============================================
# STEP 4: IDENTIFY AN ACTIONABLE RUNTIME EVENT
# ============================================

actionable = orchestrated_df.copy()

# Support boolean values stored as either True/False or strings.
if "agents_invoked" in actionable.columns:
    actionable["agents_invoked_normalized"] = (
        actionable["agents_invoked"]
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(["true", "1", "yes"])
    )
else:
    actionable["agents_invoked_normalized"] = False

actionable_events = actionable[
    actionable["agents_invoked_normalized"]
].copy()

if actionable_events.empty:
    raise RuntimeError(
        "No actionable runtime events were found."
    )

actionable_events["Cycle"] = pd.to_numeric(
    actionable_events["Cycle"],
    errors="coerce"
)

# Select the latest actionable event as the final demonstration event.
current_event = (
    actionable_events
    .sort_values(["Cycle", "Machine_ID"])
    .iloc[-1]
)

print("Actionable events available:", len(actionable_events))
print("\nSELECTED RUNTIME EVENT")
print("======================")
print("Machine:", current_event["Machine_ID"])
print("Type:", current_event["Machine_Type"])
print("Cycle:", current_event["Cycle"])
print("Fault:", current_event["Simulation_Fault"])
print("Failure probability:", current_event["failure_probability"])
print("Predicted RUL:", current_event["predicted_rul_cycles"])
print("Status:", current_event["overall_status"])
print("Maintenance decision:", current_event["maintenance_decision"])
print("Alert level:", current_event["alert_level"])


Actionable events available: 55

SELECTED RUNTIME EVENT
Machine: TURB_005
Type: Turbine
Cycle: 284
Fault: MISALIGNMENT
Failure probability: 0.9917
Predicted RUL: 15.33
Status: CRITICAL
Maintenance decision: IMMEDIATE MAINTENANCE
Alert level: CRITICAL


In [5]:
# ============================================
# STEP 5: RETRIEVE MACHINE MEMORY
# ============================================

machine_id = current_event["Machine_ID"]
current_cycle = current_event["Cycle"]

machine_history = memory_df[
    memory_df["Machine_ID"] == machine_id
].copy()

machine_history["Cycle"] = pd.to_numeric(
    machine_history["Cycle"],
    errors="coerce"
)

historical_memory = (
    machine_history[
        machine_history["Cycle"] < current_cycle
    ]
    .sort_values("Cycle", ascending=False)
    .head(5)
    .sort_values("Cycle")
    .reset_index(drop=True)
)

print(
    f"Historical memory retrieved for {machine_id}: "
    f"{len(historical_memory)} events"
)

display(
    historical_memory[
        [
            "Machine_ID",
            "Cycle",
            "Simulation_Fault",
            "failure_probability",
            "predicted_rul_cycles",
            "overall_status",
            "maintenance_decision",
            "alert_level"
        ]
    ]
)


Historical memory retrieved for TURB_005: 5 events


,Machine_ID,Cycle,Simulation_Fault,failure_probability,predicted_rul_cycles,overall_status,maintenance_decision,alert_level
0,TURB_005,254,MISALIGNMENT,0.0005,48.61,WARNING,SCHEDULE INSPECTION,WARNING
1,TURB_005,259,MISALIGNMENT,0.0007,49.96,WARNING,SCHEDULE INSPECTION,WARNING
2,TURB_005,281,MISALIGNMENT,0.6913,19.87,CRITICAL,IMMEDIATE MAINTENANCE,CRITICAL
3,TURB_005,282,MISALIGNMENT,0.2237,19.78,CRITICAL,IMMEDIATE MAINTENANCE,CRITICAL
4,TURB_005,283,MISALIGNMENT,0.0416,20.23,WARNING,SCHEDULE INSPECTION,WARNING


In [6]:
# ============================================
# STEP 6: BUILD MEMORY CONTEXT
# ============================================

if historical_memory.empty:
    memory_context = "No previous machine history was found."
else:
    memory_lines = [
        f"Historical memory for {machine_id}:"
    ]

    for _, row in historical_memory.iterrows():
        memory_lines.append(
            f"- Cycle {int(row['Cycle'])}: "
            f"status={row['overall_status']}; "
            f"failure_probability={row['failure_probability']:.4f}; "
            f"RUL={row['predicted_rul_cycles']:.2f} cycles; "
            f"fault={row['Simulation_Fault']}; "
            f"maintenance={row['maintenance_decision']}; "
            f"alert={row['alert_level']}."
        )

    memory_context = "\n".join(memory_lines)

print(memory_context)


Historical memory for TURB_005:
- Cycle 254: status=WARNING; failure_probability=0.0005; RUL=48.61 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 259: status=WARNING; failure_probability=0.0007; RUL=49.96 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 281: status=CRITICAL; failure_probability=0.6913; RUL=19.87 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 282: status=CRITICAL; failure_probability=0.2237; RUL=19.78 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 283: status=WARNING; failure_probability=0.0416; RUL=20.23 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.


In [7]:
# ============================================
# STEP 7: LOAD AND INDEX RAG KNOWLEDGE
# ============================================

import json

with open(knowledge_path, "r", encoding="utf-8") as f:
    knowledge_documents = json.load(f)

knowledge_df = pd.DataFrame(knowledge_documents)

knowledge_df["retrieval_text"] = knowledge_df.apply(
    lambda row: " ".join([
        str(row["title"]),
        str(row["machine_type"]),
        str(row["fault"]),
        str(row["content"])
    ]),
    axis=1
)

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2)
)

document_matrix = vectorizer.fit_transform(
    knowledge_df["retrieval_text"]
)

print("Knowledge documents indexed:", len(knowledge_df))


Knowledge documents indexed: 8


In [8]:
# ============================================
# STEP 8: RETRIEVE RELEVANT KNOWLEDGE
# ============================================

rag_query = (
    f"machine type {current_event['Machine_Type']} "
    f"fault {current_event['Simulation_Fault']} "
    f"status {current_event['overall_status']} "
    f"maintenance priority {current_event['maintenance_priority']} "
    f"maintenance decision {current_event['maintenance_decision']} "
    f"alert {current_event['alert_level']} "
    f"RUL {current_event['predicted_rul_cycles']} "
    f"failure probability {current_event['failure_probability']}"
)

query_vector = vectorizer.transform([rag_query])

scores = cosine_similarity(
    query_vector,
    document_matrix
).flatten()

top_k = min(3, len(knowledge_df))
top_indices = np.argsort(scores)[::-1][:top_k]

retrieved_knowledge = knowledge_df.iloc[top_indices].copy()
retrieved_knowledge["similarity_score"] = scores[top_indices]

retrieved_knowledge = (
    retrieved_knowledge
    .sort_values("similarity_score", ascending=False)
    .reset_index(drop=True)
)

print("RAG query:")
print(rag_query)

print("\nRetrieved knowledge:")
display(
    retrieved_knowledge[
        [
            "document_id",
            "title",
            "fault",
            "similarity_score"
        ]
    ]
)


RAG query:
machine type Turbine fault MISALIGNMENT status CRITICAL maintenance priority URGENT maintenance decision IMMEDIATE MAINTENANCE alert CRITICAL RUL 15.33 failure probability 0.9917

Retrieved knowledge:


,document_id,title,fault,similarity_score
0,KB_GENERAL_001,Predictive Maintenance — Warning and Critical ...,GENERAL,0.385190
1,KB_BEARING_001,Rotating Equipment — Bearing Wear,BEARING_WEAR,0.073360
2,KB_VIBRATION_001,Industrial Equipment — Excessive Vibration,VIBRATION_ANOMALY,0.056811


In [9]:
# ============================================
# STEP 9: BUILD FINAL AUGMENTED CONTEXT
# ============================================

current_context = (
    f"Machine ID: {current_event['Machine_ID']}\n"
    f"Machine Type: {current_event['Machine_Type']}\n"
    f"Cycle: {current_event['Cycle']}\n"
    f"Simulation Fault: {current_event['Simulation_Fault']}\n"
    f"Failure Probability: {current_event['failure_probability']}\n"
    f"Predicted RUL: {current_event['predicted_rul_cycles']} cycles\n"
    f"Overall Status: {current_event['overall_status']}\n"
    f"Maintenance Priority: {current_event['maintenance_priority']}\n"
    f"Maintenance Decision: {current_event['maintenance_decision']}\n"
    f"Alert Level: {current_event['alert_level']}\n"
    f"Alert Required: {current_event['alert_required']}"
)

knowledge_lines = [
    "Retrieved maintenance knowledge:"
]

for _, doc in retrieved_knowledge.iterrows():
    knowledge_lines.append(
        f"[{doc['document_id']}] {doc['title']} "
        f"(similarity={doc['similarity_score']:.3f})\n"
        f"{doc['content']}"
    )

knowledge_context = "\n".join(knowledge_lines)

final_augmented_context = (
    "CURRENT MACHINE EVENT\n"
    "=====================\n"
    f"{current_context}\n\n"
    "MACHINE MEMORY\n"
    "===============\n"
    f"{memory_context}\n\n"
    "RETRIEVED MAINTENANCE KNOWLEDGE\n"
    "================================\n"
    f"{knowledge_context}"
)

print(final_augmented_context)


CURRENT MACHINE EVENT
Machine ID: TURB_005
Machine Type: Turbine
Cycle: 284
Simulation Fault: MISALIGNMENT
Failure Probability: 0.9917
Predicted RUL: 15.33 cycles
Overall Status: CRITICAL
Maintenance Priority: URGENT
Maintenance Decision: IMMEDIATE MAINTENANCE
Alert Level: CRITICAL
Alert Required: True

MACHINE MEMORY
Historical memory for TURB_005:
- Cycle 254: status=WARNING; failure_probability=0.0005; RUL=48.61 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 259: status=WARNING; failure_probability=0.0007; RUL=49.96 cycles; fault=MISALIGNMENT; maintenance=SCHEDULE INSPECTION; alert=WARNING.
- Cycle 281: status=CRITICAL; failure_probability=0.6913; RUL=19.87 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 282: status=CRITICAL; failure_probability=0.2237; RUL=19.78 cycles; fault=MISALIGNMENT; maintenance=IMMEDIATE MAINTENANCE; alert=CRITICAL.
- Cycle 283: status=WARNING; failure_probability=0.0416; RUL=20.23 c

In [10]:
# ============================================
# STEP 10: CONNECT TO TCS MAAS LLM
# ============================================

from langchain_openai import ChatOpenAI
import httpx

api_key = os.getenv("GENAI_API_KEY")

if not api_key:
    raise RuntimeError(
        "GENAI_API_KEY is not available in this Jupyter kernel. "
        "Start Jupyter from the terminal where the key was exported."
    )

client = httpx.Client(verify=False)

llm = ChatOpenAI(
    base_url="https://genailab.tcs.in",
    model="genailab-maas-gpt-35-turbo",
    api_key=api_key,
    http_client=client
)

print("TCS MaaS LLM ready.")


TCS MaaS LLM ready.


In [11]:
# ============================================
# STEP 11: FINAL INTEGRATED LLM PROMPT
# ============================================

FINAL_SYSTEM_PROMPT = """
You are the Final Maintenance Intelligence Agent in a Smart
Manufacturing Digital Twin system.

You receive:
1. A current machine event generated by the project pipeline.
2. Historical machine memory.
3. Retrieved maintenance/domain knowledge.

Your job is to synthesize these sources into a clear maintenance
intelligence report.

CRITICAL RULES:
- Do not override the supplied maintenance decision.
- Do not override the supplied alert level.
- Do not invent measurements, history, faults, or causes.
- Distinguish observed/model-predicted evidence from possible causes.
- Use retrieved knowledge as supporting guidance.
- Safety procedures and qualified maintenance personnel remain authoritative.
- Mention historical trends when they are supported by the supplied memory.

Use exactly this structure:

1. CURRENT ASSESSMENT
2. HISTORICAL CONTEXT
3. EVIDENCE
4. RETRIEVED MAINTENANCE KNOWLEDGE
5. RECOMMENDED INSPECTION
6. OPERATOR / MAINTENANCE ACTION
7. ESCALATION CONDITIONS
"""

final_response = llm.invoke(
    [
        ("system", FINAL_SYSTEM_PROMPT),
        (
            "human",
            "Generate the final maintenance intelligence report "
            "from the following integrated context:\n\n"
            + final_augmented_context
        )
    ]
)

final_report = final_response.content

print("FINAL INTEGRATED MAINTENANCE REPORT")
print("===================================")
print(final_report)


FINAL INTEGRATED MAINTENANCE REPORT
1. CURRENT ASSESSMENT  
TURB_005 is currently in a CRITICAL status due to a MISALIGNMENT fault detected at cycle 284, with a very high failure probability of 0.9917 and a remaining useful life (RUL) of approximately 15.33 cycles. The alert level is CRITICAL, and the maintenance decision is IMMEDIATE MAINTENANCE with URGENT priority.

2. HISTORICAL CONTEXT  
Historical data shows that misalignment has been an ongoing issue for TURB_005 since at least cycle 254, initially categorized as WARNING with low failure probabilities (~0.0005 to 0.0007) and scheduled inspections. This condition worsened notably by cycle 281, when the status escalated to CRITICAL with a failure probability of 0.6913 and an RUL near 20 cycles, prompting immediate maintenance then as well. Following cycles saw fluctuations in alert and maintenance status, with cycle 283 reverting briefly to WARNING. The current event at cycle 284 signifies a sharp escalation to near-certain failur

In [12]:
# ============================================
# STEP 12: COMPARE LLM-ONLY VS RAG-ENABLED
# ============================================

# Find an existing report for the same machine/cycle.
existing_llm = llm_reports_df[
    (llm_reports_df["Machine_ID"] == current_event["Machine_ID"])
    & (llm_reports_df["Cycle"] == current_event["Cycle"])
]

existing_rag = rag_reports_df[
    (rag_reports_df["Machine_ID"] == current_event["Machine_ID"])
    & (rag_reports_df["Cycle"] == current_event["Cycle"])
]

print("CURRENT EVENT")
print("=============")
print(
    current_event[
        [
            "Machine_ID",
            "Cycle",
            "Simulation_Fault",
            "overall_status",
            "maintenance_decision"
        ]
    ].to_dict()
)

if not existing_llm.empty:
    print("\nLLM-ONLY REPORT EXISTS: YES")
    print(
        "LLM report length:",
        len(str(existing_llm.iloc[0]["llm_report"]))
    )
else:
    print("\nLLM-ONLY REPORT EXISTS: NO")

if not existing_rag.empty:
    print("RAG REPORT EXISTS: YES")
    print(
        "RAG report length:",
        len(str(existing_rag.iloc[0]["rag_report"]))
    )
else:
    print("RAG REPORT EXISTS: NO")

print(
    "\nThe final integrated report above is generated with "
    "CURRENT EVENT + MEMORY + RAG KNOWLEDGE."
)


CURRENT EVENT
{'Machine_ID': 'TURB_005', 'Cycle': 284, 'Simulation_Fault': 'MISALIGNMENT', 'overall_status': 'CRITICAL', 'maintenance_decision': 'IMMEDIATE MAINTENANCE'}

LLM-ONLY REPORT EXISTS: YES
LLM report length: 1612
RAG REPORT EXISTS: YES
RAG report length: 3517

The final integrated report above is generated with CURRENT EVENT + MEMORY + RAG KNOWLEDGE.


In [13]:
# ============================================
# STEP 13: FINAL SYSTEM HEALTH CHECK
# ============================================

checks = {
    "Orchestrator events loaded": not orchestrated_df.empty,
    "Actionable events available": not actionable_events.empty,
    "Machine memory available": not memory_df.empty,
    "Memory covers selected machine": (
        (memory_df["Machine_ID"] == machine_id).any()
    ),
    "RAG knowledge loaded": not knowledge_df.empty,
    "Knowledge retrieved": not retrieved_knowledge.empty,
    "MaaS LLM response generated": bool(
        final_report and final_report.strip()
    ),
}

print("FINAL SYSTEM HEALTH CHECK")
print("=========================")

for check, passed in checks.items():
    print(
        f"[{'PASS' if passed else 'FAIL'}] {check}"
    )

if not all(checks.values()):
    raise RuntimeError(
        "One or more final integration checks failed."
    )

print("\nAll integration checks passed.")


FINAL SYSTEM HEALTH CHECK
[PASS] Orchestrator events loaded
[PASS] Actionable events available
[PASS] Machine memory available
[PASS] Memory covers selected machine
[PASS] RAG knowledge loaded
[PASS] Knowledge retrieved
[PASS] MaaS LLM response generated

All integration checks passed.


In [14]:
# ============================================
# STEP 14: SAVE FINAL DEMONSTRATION REPORT
# ============================================

final_output_path = (
    processed_dir / "final_integrated_maintenance_report.txt"
)

with open(final_output_path, "w", encoding="utf-8") as f:
    f.write(
        "FINAL SMART MANUFACTURING MAINTENANCE REPORT\n"
        "==============================================\n\n"
    )
    f.write(
        f"Machine: {current_event['Machine_ID']}\n"
        f"Cycle: {current_event['Cycle']}\n"
        f"Fault: {current_event['Simulation_Fault']}\n"
        f"Status: {current_event['overall_status']}\n"
        f"Maintenance Decision: "
        f"{current_event['maintenance_decision']}\n"
        f"Alert Level: {current_event['alert_level']}\n\n"
    )
    f.write(final_report)

print("Final integrated report saved:")
print(final_output_path)


Final integrated report saved:
/home/labuser/Desktop/Project/3203141-Capstone_Proj/data/processed/final_integrated_maintenance_report.txt


# Final Project Architecture

The completed project can now be presented as:

```text
                         IoT Simulation
                              ↓
                       Sensor / Fault Data
                              ↓
                    Smart Twin Prediction
                       ↙           ↘
             Failure Probability   RUL
                       \           /
                        Smart Twin
                             ↓
                  ┌──────────┴──────────┐
                  ↓                     ↓
          Maintenance Agent        Alert Agent
                  └──────────┬──────────┘
                             ↓
                       Orchestrator
                             ↓
                     Actionable Event
                             ↓
              ┌──────────────┴──────────────┐
              ↓                             ↓
          Machine Memory                  RAG
       historical events          maintenance knowledge
              └──────────────┬──────────────┘
                             ↓
                            LLM
                             ↓
             Final Maintenance Intelligence
                             ↓
                     Operator / Team
```

## Key message

The project uses different technologies for different responsibilities:

- **Machine Learning:** prediction.
- **Digital Twin:** machine-state representation and predictive interpretation.
- **Agents:** specialized maintenance and alert decisions.
- **Orchestrator:** coordination.
- **Memory:** historical machine context.
- **RAG:** external/domain maintenance knowledge retrieval.
- **LLM:** contextual synthesis and natural-language explanation.

This separation makes the architecture easier to understand, test, and extend.
